## 1、local查询

### 1.1 模型配置

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from openai import OpenAI
# 实例化客户端
client = OpenAI(    
                api_key=os.getenv("openai_api_key", None),
                base_url=os.getenv("base_url", None))

In [3]:
models_list = client.models.list()

In [4]:
models_list.data

[Model(id='babbage-002', created=1626777600, object='model', owned_by='openai', permission=[{'id': 'modelperm-LwHkVFn8AcMItP432fKKDIKJ', 'object': 'model_permission', 'created': 1626777600, 'allow_create_engine': True, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}], root='babbage-002', parent=None),
 Model(id='chatgpt-4o-latest', created=1626777600, object='model', owned_by='openai', permission=[{'id': 'modelperm-LwHkVFn8AcMItP432fKKDIKJ', 'object': 'model_permission', 'created': 1626777600, 'allow_create_engine': True, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}], root='chatgpt-4o-latest', parent=None),
 Model(id='claude-3-5-haiku-20241022', created=1626777600, object='model', owned_by='vertex-ai', permission=

In [5]:
# 调用 GPT-4o-mini 模型
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "你好，好久不见!"}
    ]
)

In [6]:
# 输出生成的响应内容
print(response.choices[0].message.content)

你好！确实好久不见了！最近怎么样？有什么新鲜事分享吗？


### 1.2 查询

In [7]:
import os

import pandas as pd
import tiktoken

from graphrag.query.context_builder.entity_extraction import EntityVectorStoreKey
from graphrag.query.indexer_adapters import (
    read_indexer_covariates,
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
)
from graphrag.query.llm.oai.chat_openai import ChatOpenAI
from graphrag.query.llm.oai.embedding import OpenAIEmbedding
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.question_gen.local_gen import LocalQuestionGen
from graphrag.query.structured_search.local_search.mixed_context import (
    LocalSearchMixedContext,
)
from graphrag.query.structured_search.local_search.search import LocalSearch
from graphrag.vector_stores.lancedb import LanceDBVectorStore

In [8]:
text_unit_df = pd.read_parquet("./sushi2/output/create_final_text_units.parquet")
text_units = read_indexer_text_units(text_unit_df)

print(f"Text unit records: {len(text_unit_df)}")
text_unit_df.head()

Text unit records: 8


,id,human_readable_id,text,n_tokens,document_ids,entity_ids,relationship_ids
0,9cf74fd1963f3ea256ff5e08d48961c31a129e528543fc...,1,# 苏轼：北宋文坛的璀璨巨星\n苏轼，这位北宋时期的传奇人物，于宋景祐三年十二月十九日（10...,1200,[15f30077a469a205b2919e1b476dbae05a1c156d5d829...,"[cf842afa-7700-4563-b320-803a728e86dc, a2b9f69...","[2877a835-55ab-475c-8b84-ac8e16d7ed6b, 8cdc383..."
1,a6053b30c0cdc69a23d7b0e55aa2c29a4200cf69ef4b9d...,2,��步天下！”\n\n在欧阳修的极力称赞下，苏轼声名鹊起，一时间成为京城的焦点人物。他的每一...,1200,[15f30077a469a205b2919e1b476dbae05a1c156d5d829...,"[cf842afa-7700-4563-b320-803a728e86dc, a2b9f69...","[2877a835-55ab-475c-8b84-ac8e16d7ed6b, 96a0c2f..."
2,ad032814011ec3af535e738af040ecafe3572d4b86aeaf...,3,1077年）四月至元丰二年（1079年）三月，苏轼在徐州任知州。期间，黄河在曹村决口，洪水泛...,1200,[15f30077a469a205b2919e1b476dbae05a1c156d5d829...,"[cf842afa-7700-4563-b320-803a728e86dc, b5937d3...","[98f26f8d-cd09-4d59-947a-5c44f24902b3, 72c3d28..."
3,686ba4dd7e8a80c18f7e59e21bf5d4685c8016ec34e5c2...,4,山游览，触景生情，写下了《赤壁赋》《后赤壁赋》和《念奴娇·赤壁怀古》等千古名作，借景抒情，寄...,1200,[15f30077a469a205b2919e1b476dbae05a1c156d5d829...,"[cf842afa-7700-4563-b320-803a728e86dc, b5937d3...","[98f26f8d-cd09-4d59-947a-5c44f24902b3, a5aca70..."
4,83dd38f6b525caf8b64eb7a32627a5e614c27067736e19...,5,关注到杭州的水利问题。茅山有一条河专门容纳钱塘江潮水，盐桥有一条河专门容纳西湖水，但河道年久...,1200,[15f30077a469a205b2919e1b476dbae05a1c156d5d829...,"[cf842afa-7700-4563-b320-803a728e86dc, 174e887...","[a49b633d-cc64-4b79-9d6d-00f1ddbdf60a, 31cae5d..."


In [9]:
nodes_df = pd.read_parquet("./sushi2/output/create_final_nodes.parquet")
entitys_df = pd.read_parquet("./sushi2/output/create_final_entities.parquet")

entities = read_indexer_entities(nodes_df, entitys_df,None)

In [10]:
print(f"Entity count: {len(nodes_df)}")
nodes_df.head()

Entity count: 103


,id,human_readable_id,title,community,level,degree,x,y
0,cf842afa-7700-4563-b320-803a728e86dc,0,苏轼,0,0,89,0.0,0.0
1,a2b9f693-86f0-4fe3-bbf0-7629843dd3ec,1,北宋,3,0,3,0.0,0.0
2,f94492f1-b134-4436-aedb-071ef9ac0c80,2,眉州眉山,0,0,1,0.0,0.0
3,06b972b9-dfa0-47ce-a19b-8338abbfd986,3,苏洵,4,0,2,0.0,0.0
4,973e1a46-677a-49bb-b780-59834da4e81a,4,程氏,0,0,1,0.0,0.0


In [11]:
description_embedding_store = LanceDBVectorStore(
    collection_name="default-entity-description",
)
description_embedding_store.connect(db_uri="./sushi2/output/lancedb/")



In [12]:
relationship_df = pd.read_parquet("./sushi2/output/create_final_relationships.parquet")
relationships = read_indexer_relationships(relationship_df)

print(f"Relationship count: {len(relationship_df)}")
relationship_df.head()

Relationship count: 104


,id,human_readable_id,source,target,description,weight,combined_degree,text_unit_ids
0,2877a835-55ab-475c-8b84-ac8e16d7ed6b,0,苏轼,苏洵,苏洵是苏轼的父亲，对苏轼的成长产生了深远的影响。,18.0,91,[9cf74fd1963f3ea256ff5e08d48961c31a129e528543f...
1,8cdc3833-893b-45ad-98f7-0a040fa8e4ca,1,苏轼,程氏,程氏是苏轼的母亲，从小教育和影响了苏轼,8.0,90,[9cf74fd1963f3ea256ff5e08d48961c31a129e528543f...
2,04a3d8a7-1b05-473d-8deb-73613550f2c4,2,苏轼,范滂,范滂的事迹曾激励苏轼,5.0,90,[9cf74fd1963f3ea256ff5e08d48961c31a129e528543f...
3,96a0c2fd-020b-46f5-886d-b552ee7e912f,3,苏轼,苏辙,苏轼和苏辙是兄弟，两人在母亲去世后回乡守丧。苏辙是苏轼的弟弟，他们一起学习和成长，共同度过了...,15.0,93,[9cf74fd1963f3ea256ff5e08d48961c31a129e528543f...
4,44443f4c-9482-4c6d-bd25-a99ac86c95d1,4,苏轼,东坡肉,东坡肉是苏轼创造的美食之一,4.0,90,[9cf74fd1963f3ea256ff5e08d48961c31a129e528543f...


In [13]:
report_df = pd.read_parquet("./sushi2/output/create_final_community_reports.parquet")
reports = read_indexer_reports(report_df, nodes_df, None)

print(f"Report records: {len(report_df)}")
report_df.head()

Report records: 7


,id,human_readable_id,community,parent,level,title,summary,full_content,rank,rank_explanation,findings,full_content_json,period,size
0,52e2719a16c44bccbcddb99fed4e0094,0,0,-1,0,苏轼与北宋名人社区,该社区围绕苏轼这一中心人物展开，涉及其家庭、同事、皇帝及后代等多名北宋重要人物。苏轼作为北宋...,# 苏轼与北宋名人社区\n\n该社区围绕苏轼这一中心人物展开，涉及其家庭、同事、皇帝及后代等...,9.0,苏轼及其关联人物和事件在北宋历史和文化中具有极高的影响力。,[{'explanation': '苏轼是北宋时期的多才多艺的文人，涵盖了诗歌、书法、绘画等...,"{\n ""title"": ""苏轼与北宋名人社区"",\n ""summary"": ""...",2025-02-08,75
1,a8913497bf524bda9f9b5c6a007e5eeb,1,1,-1,0,苏轼与《和子由渑池怀旧》,本社区围绕苏轼和他的著名诗作《和子由渑池怀旧》展开。渑池是诗中提到的重要地点，而《和子由渑池...,# 苏轼与《和子由渑池怀旧》\n\n本社区围绕苏轼和他的著名诗作《和子由渑池怀旧》展开。渑池...,8.5,该社区的影响力评级较高，因为苏轼及其诗作具有广泛的历史和文化重要性，对文学和思想产生了深远的影响。,[{'explanation': '苏轼是中国历史上著名的诗人、文学家和思想家，其作品广泛传...,"{\n ""title"": ""苏轼与《和子由渑池怀旧》"",\n ""summary""...",2025-02-08,2
2,fadd168872f44fee9750435d95f0c018,2,2,-1,0,Su Shi and the Poem '题西林壁' Describing Lushan,The community revolves around the famous Chine...,# Su Shi and the Poem '题西林壁' Describing Lushan...,8.0,The impact severity rating is high due to the ...,[{'explanation': 'Su Shi is a central figure i...,"{\n ""title"": ""Su Shi and the Poem '题西林壁' De...",2025-02-08,2
3,bf7257b4ebe9466b9dc74993bd0d499a,3,3,-1,0,Northern Song Dynasty and its Capital Bianjing,This community revolves around the Northern So...,# Northern Song Dynasty and its Capital Bianji...,8.5,The impact severity rating is high due to the ...,"[{'explanation': 'Bianjing, known today as Kai...","{\n ""title"": ""Northern Song Dynasty and its...",2025-02-08,3
4,bc30554627ea439a8eccef0285ce9ca2,4,4,-1,0,The Su Family and Their Literary Legacy,"The community revolves around the Su family, p...",# The Su Family and Their Literary Legacy\n\nT...,7.5,The impact severity rating is high due to the ...,"[{'explanation': 'Su Xun, the father of Su Shi...","{\n ""title"": ""The Su Family and Their Liter...",2025-02-08,4


In [14]:
llm = ChatOpenAI(
    api_key=os.getenv("openai_api_key", None),
    model="gpt-4o",
    api_base=os.getenv("base_url", None),
    api_type=OpenaiApiType.OpenAI,  
    max_retries=20,
)

token_encoder = tiktoken.get_encoding("cl100k_base")

text_embedder = OpenAIEmbedding(
    api_key=os.getenv("openai_api_key", None),
    api_base=os.getenv("base_url", None),
    api_type=OpenaiApiType.OpenAI,
    model="text-embedding-3-large",
    deployment_name="text-embedding-3-large",
    max_retries=20,
)

In [15]:
context_builder = LocalSearchMixedContext(
    community_reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    covariates=None,
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=EntityVectorStoreKey.ID,  
    text_embedder=text_embedder,
    token_encoder=token_encoder,
)

In [16]:
local_context_params = {
    "text_unit_prop": 0.5,
    "community_prop": 0.1,
    "conversation_history_max_turns": 5,
    "conversation_history_user_turns_only": True,
    "top_k_mapped_entities": 10,
    "top_k_relationships": 10,
    "include_entity_rank": True,
    "include_relationship_weight": True,
    "include_community_rank": True,
    "return_candidate_context": True,
    "embedding_vectorstore_key": EntityVectorStoreKey.ID,  
    "max_tokens": 12_000, 
}

llm_params = {
    "max_tokens": 2_000, 
    "temperature": 0.0,
}

In [17]:
search_engine = LocalSearch(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
    response_type="multiple paragraphs", 
)

In [18]:
result = await search_engine.asearch("请帮我介绍下苏轼")

In [19]:
from IPython.display import Markdown, display

display(Markdown(result.response))

### 苏轼：北宋文坛的传奇人物  

苏轼（1037年－1101年），字子瞻，号东坡居士，是北宋时期著名的文学家、政治家、书法家、画家和诗人。作为“唐宋八大家”之一，他在多个领域皆取得非凡成就，是中国文化史上最具影响力的人物之一。他的作品充满了对人生哲理的感悟和对自然景观的深刻描绘，代表作包括《赤壁赋》、《水调歌头·明月几时有》和《念奴娇·赤壁怀古》等 [Data: Entities (0); Reports (0), (5)]。

---

### 家世与成长  

苏轼出生于四川眉州眉山，出身书香门第。他的父亲苏洵和弟弟苏辙同为著名文学家，与苏轼合称为“三苏” [Data: Entities (0, 3, 6); Relationships (0, 3)]. 苏洵见证了苏轼的早年成长，对其学识和品格的形成有深远的影响。而母亲程氏则通过讲述东汉名士范滂的故事激励苏轼，播下了其正直和担当的精神种子 [Data: Entities (0, 1); Sources (0)]. 在父母的教育下，苏轼少年时期博览群书，打下了深厚的文学功底。

---

### 学业与仕途的起点  

苏轼在1048年随父亲和弟弟从家乡沿长江赴京参加科举考试，他的文章脱颖而出，得到主考官欧阳修和小试官梅尧臣的高度认可。这让苏轼一举成名，为其仕途奠定了基础 [Data: Reports (0); Entities (6, 12, 13); Relationships (6, 7, 16)]. 随后的仕途中，他在全国多个地区担任地方官员，以其大量的民生举措和工程赢得了百姓的称赞。例如，他在杭州主持修建了“苏公堤”，改善了西湖地区的水利问题 [Data: Reports (0), (4); Entities (25, 65); Relationships (20, 65)]。

---

### 乌台诗案与贬谪生涯  

苏轼因直言时政和诗文触犯了权贵而卷入了“乌台诗案”，被贬至黄州，使其政治生涯转折。尽管遭遇重重挫折，他并未放弃创作，反而在贬谪期间创作了诸多佳作，包括《赤壁赋》和《后赤壁赋》等 [Data: Reports (0); Entities (39, 55); Relationships (39, 40, 55), Sources (3)]. 他在后来的惠州和海南岛的贬谪生涯中，仍努力传播文化，甚至创建学校，培养后人 [Data: Reports (5); Entities (80, 91, 92)]。

---

### 文化贡献与美食创造  

苏轼不仅以文学闻名，还在艺术和饮食方面展现了独特的才华。他的书法被誉为“苏字”的典范，而画作同样名动当时。此外，苏东坡还凭借自己的创造力为中国饮食文化留下了不少遗产，如东坡肉、东坡饼 [Data: Entities (0); Relationships (4, 5)]. 由此可见，他不仅是文化传承的重要象征，其创造力也深入到生活的点滴。

---

### 后世影响与纪念  

苏轼的一生充满传奇色彩，且对于文学和文化的广泛贡献影响深远。他的家族后人如孙子苏过也继承其文化遗产，世代传承 [Data: Reports (0); Entities (88)]. 此外，与苏轼相关的遗迹（如儋州的东坡村、东坡井）则成为后人缅怀其生活和成就的重要场所 [Data: Entities (91, 92, 93); Reports (5)]. 宋高宗和宋孝宗也追赠其为太师并谥号“文忠”，以表彰其功绩与文学成就 [Data: Sources (5)]。

---

### 总结  

苏轼的一生是文化与政治交织的精彩历程。他在文学、艺术、政治和生活层面皆有深刻建树，尽管一生历经坎坷，却以豁达的态度面对。他不仅是北宋文坛的瑰宝之一，更成为后世数百年来的文化偶像和精神丰碑。

## 2、global查询

In [20]:
import pandas as pd
import tiktoken

from graphrag.query.indexer_adapters import (
    read_indexer_communities,
    read_indexer_entities,
    read_indexer_reports,
)
from graphrag.query.llm.oai.chat_openai import ChatOpenAI
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.structured_search.global_search.community_context import (
    GlobalCommunityContext,
)
from graphrag.query.structured_search.global_search.search import GlobalSearch

In [21]:
community_df = pd.read_parquet("./sushi2/output/create_final_communities.parquet")
communities = read_indexer_communities(community_df, nodes_df, report_df)


In [22]:
context_builder = GlobalCommunityContext(
    community_reports=reports,
    communities=communities,
    entities=entities,  
    token_encoder=token_encoder,
)

In [23]:
context_builder_params = {
    "use_community_summary": False,  
    "shuffle_data": True,
    "include_community_rank": True,
    "min_community_rank": 0,
    "community_rank_name": "rank",
    "include_community_weight": True,
    "community_weight_name": "occurrence weight",
    "normalize_community_weight": True,
    "max_tokens": 12_000,  
    "context_name": "Reports",
}

map_llm_params = {
    "max_tokens": 1000,
    "temperature": 0.0,
    "response_format": {"type": "json_object"},
}

reduce_llm_params = {
    "max_tokens": 2000, 
    "temperature": 0.0,
}

In [24]:
search_engine = GlobalSearch(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    max_data_tokens=12_000,  
    map_llm_params=map_llm_params,
    reduce_llm_params=reduce_llm_params,
    allow_general_knowledge=False, 
    json_mode=True,  
    context_builder_params=context_builder_params,
    concurrent_coroutines=32,
    response_type="multiple paragraphs",
)

In [25]:
result = await search_engine.asearch("请帮我介绍下苏轼")

In [26]:
display(Markdown(result.response))

### 苏轼的简介与成就

苏轼（1037年—1101年），字子瞻，号东坡居士，北宋时期最杰出的文人之一，以卓越的才华横跨诗歌、词赋、书法、绘画等多个文学与艺术领域。他的代表作包括《赤壁赋》、《后赤壁赋》和《念奴娇·赤壁怀古》等，这些作品不仅展现其深厚的文学修养，也反映了他对人生、自然及历史的深刻思考 [Data: Reports (0)]。

苏轼出身书香门第，其家庭对他的成长与成才有着根本性的影响。他的父亲苏洵是著名文学家，其母程氏则因知书达理而闻名。优渥的家庭环境培养了苏轼的文学天赋和治政能力，也塑造了他宽厚豁达的人格 [Data: Reports (0)]。

---

### 苏轼的政治生涯与贬谪经历

苏轼在政治生涯中曾担任多个重要职务，例如杭州知州。他在杭州期间主持修建了著名的苏公堤，并实施了多项水利和民生工程，充分展现了其治政才能。这些政绩使他深受百姓爱戴 [Data: Reports (0)]。

然而，苏轼的仕途并非一帆风顺。他因“乌台诗案”被贬至黄州，此后又被流放至惠州和海南岛等偏远地区。这些多次贬谪对他的事业造成较大打击，但他却从未因此消沉。相反，苏轼在逆境中依然持续创作，留下了大量脍炙人口的作品，并且与当地百姓建立深厚情谊 [Data: Reports (0)]。

---

### 苏轼的文化与生活影响

苏轼的贡献远远超越了他所在的时代。在文学方面，他为后世留下了无数经典佳作，影响了整个中国文学发展史。此外，他还以开朗豁达的生活态度创造了许多经典美食，如东坡肉、东坡饼，它们至今仍广受赞誉 [Data: Reports (0)]。苏轼的文化遗产也得以通过后代传承，他的孙子苏过等人继续传播并发展了苏轼的思想和成就 [Data: Reports (0)]。

---

### 结语

苏轼的一生丰富多彩，既有文学上的辉煌成就，也有坎坷的政治经历。他始终以旷达的哲学观拥抱生活、战胜逆境，为后世留下了不可磨灭的精神遗产。他的作品、事迹与人格魅力在中国历史与文化中占据了不可替代的重要地位 [Data: Reports (0)]。

## 3、可视化